# Project: RAG Research Assistant

A full **Retrieval-Augmented Generation (RAG)** pipeline with conversation memory.

**Pipeline:**
```
Documents
    │  RecursiveCharacterTextSplitter → chunks
    │  OpenAIEmbeddings → vectors
    ▼
Chroma (vector store)  ← persist_directory
    │
    │  User question
    ▼
Retriever (similarity search OR MultiQueryRetriever)
    │  retrieved docs
    ▼
ChatPromptTemplate (system + MessagesPlaceholder + context + question)
    │
    ▼
ChatOpenAI  →  str response OR structured ResearchResponse
    │
    ▼
Session memory (InMemoryChatMessageHistory per session_id)
```

Topics covered:
- `RecursiveCharacterTextSplitter` for chunking
- `OpenAIEmbeddings` + `Chroma` for vector storage
- Basic retriever vs. `MultiQueryRetriever` (advanced)
- RAG chain with conversation memory
- Structured output (`with_structured_output`) for typed responses

In [ ]:
import shutil, json
from typing import List, Dict, Optional
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from pydantic import BaseModel, Field

# Clean up any leftover DB from a previous run
shutil.rmtree("./research_db", ignore_errors=True)

## 1. Define the Structured Response Schema

In [ ]:
class ResearchResponse(BaseModel):
    """Structured response from the research assistant."""
    answer: str = Field(description="The answer to the question")
    confidence: str = Field(description="high, medium, or low based on source quality")
    sources: List[str] = Field(description="List of source documents used")
    key_quotes: List[str] = Field(description="Relevant quotes from sources", default=[])
    follow_up_questions: List[str] = Field(description="Suggested follow-up questions")

## 2. The Research Assistant Class

Three components are initialized in `__init__`:
1. **`OpenAIEmbeddings`** — converts text to dense vectors
2. **`RecursiveCharacterTextSplitter`** — breaks documents into overlapping chunks  
   `chunk_overlap` ensures context isn't lost at chunk boundaries
3. **`Chroma`** — stores and searches embedding vectors on disk (`persist_directory`)

In [ ]:
class AIResearchAssistant:
    def __init__(self, persist_directory: str = "./research_db", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        # Splitter — separators tried in order; falls back to smaller units
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

        # Chroma — persists to disk automatically
        self.vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="research_docs",
        )

        self.session_store: Dict[str, InMemoryChatMessageHistory] = {}
        print(f"Assistant initialized. Indexed chunks: {self.vectorstore._collection.count()}")

    def add_text(self, text: str, source: str, metadata: dict = None) -> int:
        doc = Document(page_content=text, metadata={"source": source, **(metadata or {})})
        chunks = self.splitter.split_documents([doc])
        for c in chunks:
            c.metadata["indexed_at"] = datetime.now().isoformat()
        self.vectorstore.add_documents(chunks)
        print(f"Added {len(chunks)} chunks from '{source}'")
        return len(chunks)

    def _build_retriever(self, use_advanced: bool = False):
        """Base = similarity search. Advanced = MultiQueryRetriever (LLM generates variants)."""
        base = self.vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
        if not use_advanced:
            return base
        return MultiQueryRetriever.from_llm(retriever=base, llm=self.llm)

    def _format_docs(self, docs) -> str:
        if not docs:
            return "No relevant documents found."
        return "\n\n---\n\n".join(
            f"[Source: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}" for d in docs
        )

    def _get_history(self, session_id: str) -> BaseChatMessageHistory:
        if session_id not in self.session_store:
            self.session_store[session_id] = InMemoryChatMessageHistory()
        return self.session_store[session_id]

    def ask(self, question: str, session_id: str = "default", use_advanced: bool = True) -> str:
        """Ask a question — returns a plain string response."""
        history = self._get_history(session_id)
        retriever = self._build_retriever(use_advanced)
        docs = retriever.invoke(question)
        context = self._format_docs(docs)

        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an AI Research Assistant. Answer questions ONLY from the context below.\n"
             "Rules:\n1. Only use context docs\n2. Say if context lacks the answer\n3. Cite sources"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "Context:\n{context}\n\nQuestion: {question}"),
        ])

        chain = prompt | self.llm | StrOutputParser()
        response = chain.invoke({"context": context, "question": question, "history": history.messages[-10:]})

        history.add_message(HumanMessage(content=question))
        history.add_message(AIMessage(content=response))
        return response

    def ask_structured(self, question: str, session_id: str = "default") -> ResearchResponse:
        """Ask a question — returns a typed ResearchResponse."""
        history = self._get_history(session_id)
        structured_llm = self.llm.with_structured_output(ResearchResponse)
        retriever = self._build_retriever(use_advanced=True)
        docs = retriever.invoke(question)
        context = self._format_docs(docs)
        sources = list(set(d.metadata.get("source", "Unknown") for d in docs))

        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an AI Research Assistant. Analyze the provided documents and return a structured response.\n"
             "Only use information from the context. Set confidence appropriately."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "Context:\n{context}\n\nAvailable sources: {sources}\n\nQuestion: {question}"),
        ])

        chain = prompt | structured_llm
        response = chain.invoke({
            "context": context,
            "question": question,
            "sources": ", ".join(sources),
            "history": history.messages[-10:],
        })

        history.add_message(HumanMessage(content=question))
        history.add_message(AIMessage(content=response.answer))
        return response

    def list_sources(self) -> List[str]:
        results = self.vectorstore._collection.get()
        return sorted(set(m["source"] for m in results.get("metadatas", []) if m and "source" in m))

## 3. Add Documents & Build the Index

In [ ]:
assistant = AIResearchAssistant()

assistant.add_text("""
Attention Mechanisms in Neural Networks

The attention mechanism was introduced in "Attention Is All You Need" by Vaswani et al. (2017).
It allows models to focus on relevant parts of the input when generating output.

Key concepts:
- Query, Key, Value (QKV) triplets
- Scaled dot-product attention
- Multi-head attention for parallel processing

The transformer architecture has become the foundation for modern NLP models including BERT, GPT, and T5.
""", source="attention_mechanisms.pdf")

assistant.add_text("""
Retrieval-Augmented Generation (RAG)

RAG combines retrieval systems with generative models. First introduced by Lewis et al. (2020),
RAG addresses the limitation of LLMs being limited to their training data.

Components of a RAG system:
1. Document store with vector embeddings
2. Retriever to find relevant documents
3. Generator (LLM) to produce responses

Benefits include reduced hallucination, up-to-date information, and source attribution.
""", source="rag_survey.pdf")

assistant.add_text("""
LangChain and LangGraph Framework Overview

LangChain is an open-source framework for building LLM applications.
Key features include modular components, integration with 50+ LLM providers, and built-in RAG utilities.

LangGraph extends LangChain for stateful applications with graph-based state management,
support for cycles and loops, and human-in-the-loop workflows.
""", source="langchain_docs.md")

print(f"\nTotal indexed chunks: {assistant.vectorstore._collection.count()}")
print(f"Sources: {assistant.list_sources()}")

## 4. Basic vs Advanced Retrieval

- **Basic**: single similarity search query → k closest chunks
- **Advanced (MultiQueryRetriever)**: LLM generates multiple search query variants → broader recall

In [ ]:
# Basic retrieval — single query
print("--- Basic Retrieval ---")
answer = assistant.ask("What is RAG and what are its benefits?", "basic_test", use_advanced=False)
print(answer[:400])

## 5. Structured Response

In [ ]:
print("--- Structured Response ---")
result = assistant.ask_structured("What is the attention mechanism?", "struct_test")

print(f"Answer:     {result.answer[:200]}")
print(f"Confidence: {result.confidence}")
print(f"Sources:    {result.sources}")
print(f"Key quotes: {result.key_quotes[:2]}")
print(f"Follow-ups: {result.follow_up_questions}")

## 6. Multi-turn Conversation Memory

In [ ]:
session = "multi_turn"

q1 = "What are the components of RAG?"
print(f"Q1: {q1}")
r1 = assistant.ask_structured(q1, session)
print(f"A1: {r1.answer[:200]}\n")

# Follow-up refers to previous answer — tests memory
q2 = "How does the first component you mentioned work?"
print(f"Q2: {q2}")
r2 = assistant.ask_structured(q2, session)
print(f"A2: {r2.answer[:200]}\n")

print(f"Messages in session: {len(assistant._get_history(session).messages)}")

# Cleanup
shutil.rmtree("./research_db", ignore_errors=True)